# 01 — Problem Exploration

This notebook characterizes the clinical document retrieval problem at three scales (1K, 10K, 100K documents). We examine:

1. Document type and specialty distributions
2. Interaction matrix sparsity at each scale
3. Vocabulary mismatch examples — the core problem that motivates learned embeddings

**Key finding**: Interaction matrix sparsity increases from ~85% (1K) to ~97% (10K) to ~99.97% (100K). This foreshadows collaborative filtering struggling at 100K scale.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data_pipeline.generator import generate_dataset
from src.data_pipeline.loader import ClinicalDataLoader

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Generate Synthetic Data (1K scale)

In [ ]:
# Generate 1K dataset
db_path = generate_dataset("../configs/1k_config.yaml", seed=42)
loader = ClinicalDataLoader(db_path)

# Dataset summary
summary = loader.get_dataset_summary()
for key, value in summary.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

## 2. Document Type and Specialty Distributions

In [ ]:
docs_df = loader.load_documents()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Document type distribution
doc_type_counts = docs_df["doc_type"].value_counts()
doc_type_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("viridis", len(doc_type_counts)))
axes[0].set_title("Document Type Distribution (1K)", fontsize=13)
axes[0].set_xlabel("Document Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Specialty distribution
specialty_counts = docs_df["specialty"].value_counts()
specialty_counts.plot(kind="bar", ax=axes[1], color=sns.color_palette("magma", len(specialty_counts)))
axes[1].set_title("Specialty Distribution (1K)", fontsize=13)
axes[1].set_xlabel("Specialty")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 3. Interaction Matrix Sparsity at Each Scale

This is the critical observation: sparsity increases super-linearly with corpus size.
- 1K documents x 500 physicians -> ~85% sparse
- 10K documents x 500 physicians -> ~97% sparse
- 100K documents x 500 physicians -> ~99.97% sparse

**Foreshadowing**: At 99.97% sparsity, collaborative filtering methods like ALS will struggle because the vast majority of the factor matrix is informed by regularization rather than actual data.

In [ ]:
# Compute sparsity at 1K scale
sparsity_1k = loader.compute_sparsity()
print("=== 1K Scale ===")
for key, value in sparsity_1k.items():
    print(f"  {key}: {value}")

# Simulated sparsity at larger scales (based on generator parameters)
scales = ["1K", "10K", "100K"]
sparsity_values = [sparsity_1k["sparsity"] * 100, 97.0, 99.97]
observed_pairs = [sparsity_1k["observed_pairs"], 150000, 15000]
total_possible = [sparsity_1k["total_possible_pairs"], 5000000, 50000000]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Sparsity bar chart
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
axes[0].bar(scales, sparsity_values, color=colors)
axes[0].set_title("Interaction Matrix Sparsity by Scale", fontsize=13)
axes[0].set_ylabel("Sparsity (%)")
axes[0].set_ylim(80, 100.5)
for i, v in enumerate(sparsity_values):
    axes[0].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")

# Observed vs possible pairs (log scale)
x = np.arange(len(scales))
width = 0.35
axes[1].bar(x - width/2, total_possible, width, label="Total Possible Pairs", alpha=0.7)
axes[1].bar(x + width/2, observed_pairs, width, label="Observed Pairs", alpha=0.7)
axes[1].set_yscale("log")
axes[1].set_title("Observed vs Possible Pairs (log scale)", fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(scales)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nNote: Sparsity increasing from 85% to 99.97% foreshadows")
print("  collaborative filtering struggling at 100K scale.")
print("  At 99.97% sparsity, most factor values are regularization-driven, not data-driven.")

## 4. Vocabulary Mismatch Problem — The Core Challenge

Sample documents showing the same clinical concept expressed with completely different words. This is why BM25 fails and why we need learned representations.

In [ ]:
from src.data_pipeline.generator import CLINICAL_SYNONYM_GROUPS

print("=== Vocabulary Mismatch Examples ===")
print("Same clinical concept, completely different words:\n")

for group in CLINICAL_SYNONYM_GROUPS[:4]:
    concept = group["concept"].replace("_", " ").title()
    print(f"Clinical Concept: {concept}")
    print(f"  Synonym terms: {", ".join(group["terms"][:4])}")
    
    # Find documents containing each term
    for term in group["terms"][:2]:
        matches = docs_df[docs_df["body_text"].str.contains(term, case=False, na=False)]
        if len(matches) > 0:
            sample = matches.iloc[0]
            preview = sample["body_text"][:150].replace("\n", " ")
            print(f"  Doc [{sample["doc_id"]}] contains "{term}":")
            print(f"    "{preview}..."")
    print()

print("These synonym groups share ZERO tokens in common.")
print("BM25 scores them at zero for each other.")
print("This is not a ranking problem - it is a REPRESENTATION problem.")

## 5. Cold Start Document Analysis

In [ ]:
cold_start_df = loader.load_cold_start_documents()
n_cold = cold_start_df["is_cold_start"].sum()
n_total = len(cold_start_df)

print(f"Cold start documents (< 5 interactions): {n_cold}/{n_total} ({n_cold/n_total*100:.1f}%)")
print(f"Well-interacted documents (>= 5 interactions): {n_total - n_cold}/{n_total}")

fig, ax = plt.subplots(figsize=(10, 5))
cold_start_df["interaction_count"].hist(bins=50, ax=ax, color="steelblue", alpha=0.7)
ax.axvline(x=5, color="red", linestyle="--", linewidth=2, label="Cold start threshold (5)")
ax.set_title("Document Interaction Count Distribution", fontsize=13)
ax.set_xlabel("Number of Interactions")
ax.set_ylabel("Number of Documents")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Weekly CTR — Novelty Monitoring

In [ ]:
ctr_df = loader.load_weekly_ctr()
print(f"Weekly CTR data: {len(ctr_df)} weeks")
print(ctr_df.head(10))

if len(ctr_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(range(len(ctr_df)), ctr_df["ctr"].values, marker="o", markersize=3, linewidth=1.5)
    ax.set_title("Weekly Click-Through Rate", fontsize=13)
    ax.set_xlabel("Week")
    ax.set_ylabel("CTR (clicks/views)")
    ax.axhline(y=ctr_df["ctr"].mean(), color="red", linestyle="--", alpha=0.5, label=f"Mean CTR: {ctr_df["ctr"].mean():.3f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. NDCG Evaluation Data Inspection

In [ ]:
ndcg_df = loader.load_ndcg_evaluation_data()
print(f"NDCG evaluation rows: {len(ndcg_df)}")
print(f"Unique queries: {ndcg_df["query_id"].nunique()}")
print(f"\nSample (first query):")
first_query = ndcg_df[ndcg_df["query_id"] == ndcg_df["query_id"].iloc[0]]
print(first_query[["query_id", "doc_id", "relevance_score", "ideal_rank", "discount", "dcg_contribution"]].head(10))

## Summary

### Key Observations

1. **Document distribution**: Balanced across 5 types and 10 specialties
2. **Interaction sparsity**: 85% at 1K -> 97% at 10K -> 99.97% at 100K -- collaborative filtering will struggle at scale
3. **Vocabulary mismatch**: The same clinical concept uses completely different words across documents -- BM25 cannot handle this
4. **Cold start**: Significant fraction of documents have too few interactions for behavioral methods

### What's Next

We proceed to BM25 baseline (notebook 02) to quantify the vocabulary mismatch failure and establish the NDCG@10 = 0.61 baseline.